In [1]:
import pandas as pd
import numpy as np

df_fake = pd.read_csv("Fake.csv")
df_true = pd.read_csv("True.csv")

df_fake["class"] = 0
df_true["class"] = 1

# Manual testing rows
df_fake_manual_testing = df_fake.tail(10)
df_true_manual_testing = df_true.tail(10)

df_fake = df_fake.iloc[:-10]
df_true = df_true.iloc[:-10]

df_manual_testing = pd.concat([df_fake_manual_testing, df_true_manual_testing], axis=0)
df_manual_testing.to_csv("manual_testing.csv", index=False)

# Merge datasets
df_merge = pd.concat([df_fake, df_true], axis=0)
df_merge = df_merge.sample(frac=1).reset_index(drop=True)


In [2]:
import re
import string

def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

df_merge['text'] = df_merge['title'] + " " + df_merge['text']
df_merge = df_merge.drop(columns=['title', 'subject', 'date'])

df_merge['text'] = df_merge['text'].apply(wordopt)


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

x = df_merge['text']
y = df_merge['class']

xv = TfidfVectorizer()
xv_train = xv.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(xv_train, y, test_size=0.25)



In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

LR = LogisticRegression()
LR.fit(x_train, y_train)

# Prediction and Accuracy
pred_LR = LR.predict(x_test)
print("Accuracy:", accuracy_score(y_test, pred_LR))


Accuracy: 0.9859180035650624


In [8]:
def manual_testing(news):
    news = wordopt(news)
    vectorized = xv.transform([news])
    prediction = LR.predict(vectorized)
    return "Fake News" if prediction[0] == 0 else "Real News"

# Manual input
news_input = input("Enter news text:\n")
result = manual_testing(news_input)
print("Prediction:", result)


Enter news text:
 the cat was found flying


Prediction: Fake News
